This notebook reads lidar and camera data from UR bag and converts it into a format that can be used by the evaluate_flow_calibration.py script

In [18]:
import numpy as np
import cv2
from pathlib import Path
from tqdm import tqdm
import matplotlib.pyplot as plt
from PIL import Image

from mcap_protobuf.decoder import DecoderFactory
from mcap.reader import make_reader

from tkCloudProtoConverter import protoCloudToNumpy, protoCloudToPcdFile

In [2]:
DATA_FOLDER = Path("/data/ur")
LIDAR_TOPIC = "/lidar_fc/cloud"
CAMERA_TOPIC = "/camera_fl/image"

In [17]:
f = open(DATA_FOLDER / "short.mcap", "rb")
reader = make_reader(f, decoder_factories=[DecoderFactory()])

last_pcd = None
last_img = None
for schema, channel, message, proto_msg in tqdm(
        reader.iter_decoded_messages(topics=[LIDAR_TOPIC, CAMERA_TOPIC])
    ):
    if channel.topic == CAMERA_TOPIC:
        nparr = np.frombuffer(proto_msg.data, np.uint8)
        img = cv2.imdecode(nparr, cv2.IMREAD_COLOR)
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        last_img = img
    elif channel.topic == LIDAR_TOPIC:
        last_pcd = protoCloudToPcdFile(proto_msg)

0it [00:00, ?it/s]

785it [04:20,  3.01it/s] 


In [19]:
# save image and pointcloud
if last_img is not None:
    img = Image.fromarray(last_img, 'RGB')
    img.save((DATA_FOLDER / "camera/0000.png").__str__())
if last_pcd is not None:
    with open(DATA_FOLDER / "lidar/0000.pcd", "wb") as f:
        f.write(last_pcd)

# Test load PCD

Reading PCD file and comparing it to the original format to ensure that the data is being saved correctly.

In [6]:
with open(DATA_FOLDER / "lidar/0000.pcd", "rb") as f:
    pcd = f.read()

bin_data = pcd[pcd.find(b"binary\n")+7:]


In [16]:
bin_points = np.frombuffer(bin_data, dtype=np.float32)
bin_points.reshape(-1, 3).shape

(38053, 3)

In [4]:
points = protoCloudToNumpy(proto_msg)

In [7]:
points.shape

(38053, 6)